In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted


def plot_averaged_layers(
    chunks_to_plot,
    base_path,
    sample_name,
    plot_config=None,
):
    """
    Plot layer-wise averaged attention with full plotting control.
    """

    # ---------------- Config ----------------
    plot_config = plot_config or {}
    color_map = plot_config.get("colors", {})
    rename_map = plot_config.get("rename", {})
    highlight_segments = plot_config.get("highlight_segments", {})
    base_alpha = plot_config.get("base_alpha", 0.35)
    highlight_alpha = plot_config.get("highlight_alpha", 0.9)
    token_colors = plot_config.get("token_colors", {})
    linewidth = plot_config.get("linewidth", 2.0)
    highlight_lw = plot_config.get("highlight_linewidth", 3.5)

    legend_cfg = plot_config.get("legend", {})
    show_spines = plot_config.get("show_spines", True)

    xlabel = plot_config.get("xlabel", "Step")
    ylabel = plot_config.get("ylabel", "Mean attention")

    # ---------------- Chunk names ----------------
    chunk_names = [
        'im_start_0', 'user', 'blank', 'image', 'post_im_newline', 'text',
        'instruction', 'im_end', 'im_start_1', 'assistant',
        'post_assistant_newline', 'previously_generating_tokens',
        'currently_generating_token'
    ]

    attention_progression = {
        f"{tgt}__attends_to__{src}": []
        for i, src in enumerate(chunk_names)
        for tgt in chunk_names[i:]
    }

    # ---------------- Load data ----------------
    generated_tokens = None

    for layer_folder in natsorted(os.listdir(base_path)):
        if layer_folder == "logs":
            continue

        file_path = os.path.join(
            base_path, layer_folder, "attention_progression", sample_name
        )
        if not os.path.exists(file_path):
            continue

        data = np.load(file_path, allow_pickle=True)

        if generated_tokens is None:
            generated_tokens = list(data["generated_tokens"])
            generated_tokens[-1] = "<EOS>"

        for k in attention_progression:
            if k not in data:
                continue
            arr = np.array(data[k], dtype=object)
            arr = np.where(arr == None, np.nan, arr)
            attention_progression[k].append(arr.tolist())

    if generated_tokens is None:
        raise ValueError(f"No data found for {sample_name}")

    # ---------------- Mean ----------------
    attention_progression_avg = {
        k: np.nanmean(np.array(v, dtype=float), axis=0) if len(v) else np.array([])
        for k, v in attention_progression.items()
    }

    # ---------------- Plot ----------------
    num_tokens = len(generated_tokens)
    x = np.arange(num_tokens)

    fig, ax = plt.subplots(figsize=(16, 9))

    for chunk in chunks_to_plot:
        y = attention_progression_avg.get(chunk)
        if y is None or not y.size:
            continue

        label = rename_map.get(chunk, chunk.replace("__attends_to__", " → "))
        color = color_map.get(chunk)

        line, = ax.plot(
            x,
            y,
            label=label,
            color=color,
            alpha=base_alpha,
            linewidth=linewidth,
            marker="o",
            zorder=5,
        )
        color = line.get_color()

        # ---- Highlight segments ----
        for start, end in highlight_segments.get(chunk, []):
            start = max(0, start)
            end = min(num_tokens - 1, end)
            ax.plot(
                x[start:end + 1],
                y[start:end + 1],
                color=color,
                linewidth=highlight_lw,
                alpha=highlight_alpha,
                zorder=10,
            )

    # ---------------- Bottom axis: step + shifted tokens ----------------
    bottom_tokens = [""] + generated_tokens[:-1]
    
#     bottom_labels = [
#     f"{i}\n{tok}" for i, tok in enumerate(bottom_tokens)
# ]
    bottom_labels = []
    for i, tok in enumerate(bottom_tokens):
        display_tok = tok if tok else r'\n'
        bottom_labels.append(f"{i}\n{display_tok}")


    ax.set_xticks(x)
    ax.set_xticklabels(bottom_labels, rotation=45, ha="right")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # ---------------- Top axis: tokens only ----------------
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(x)
    ax_top.set_xticklabels(generated_tokens, rotation=45, ha="left")

    # ---- Token coloring (both axes) ----
    for idx, tick in enumerate(ax.get_xticklabels()):
        if idx in token_colors:
            tick.set_color(token_colors[idx])
            tick.set_fontweight("bold")

    for idx, tick in enumerate(ax_top.get_xticklabels()):
        if idx in token_colors:
            tick.set_color(token_colors[idx])
            tick.set_fontweight("bold")

    # ---------------- Legend ----------------
    ax.legend(
        frameon=False,
        ncol=legend_cfg.get("ncols", 1),
        fontsize=legend_cfg.get("fontsize", 11),
        loc=legend_cfg.get("loc", "best"),
        bbox_to_anchor=legend_cfg.get("bbox_to_anchor", None),
    )

    ax.grid(alpha=0.3)

    # ---------------- Spines ----------------
    if not show_spines:
        for spine in ax.spines.values():
            spine.set_visible(False)
        for spine in ax_top.spines.values():
            spine.set_visible(False)
    
    plt.tight_layout()
    # plt.show()
    # plt.close()

    return attention_progression_avg, plt


In [ ]:
sample_name = 'fruit_math_boosting__image-actual__fruit_math_prompt__fruit-orange__math-14__id-0.npz'

In [ ]:
plot_config = {
    "colors": {
        "currently_generating_token__attends_to__image": "#FFA500",
        "currently_generating_token__attends_to__text": "#267E59",
        "currently_generating_token__attends_to__previously_generating_tokens": "#C01BA7" ,
        "currently_generating_token__attends_to__instruction": "#D25B5B",
        "currently_generating_token__attends_to__im_start_0": "#2A4B89" ,

    },
    "rename": {
        "currently_generating_token__attends_to__image": "CGT → Image",
        "currently_generating_token__attends_to__text": "CGT → Text",
    },
    "highlight_segments": {
        # "currently_generating_token__attends_to__image": [(3, 8)],
        # "currently_generating_token__attends_to__instruction": [(6, 9)],
        # "currently_generating_token__attends_to__text": [(15,22)],
        # "currently_generating_token__attends_to__im_start_0": [(0,22)],
        # "currently_generating_token__attends_to__previously_generating_tokens": [(0,22)],
    },
    "base_alpha": 0.3,
    "highlight_alpha": 0.95,
    # "token_colors": {
    #     7: "#FFA500",
    #     17: "#267E59",
    #     18: "#267E59",
    #     19: "#267E59",
    #     20: "#267E59",
    #     21: "#267E59",
    # }
}

plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/boosting_test_q3b_precomputed_vanilla",
    sample_name=sample_name,
    plot_config=plot_config,
)


In [ ]:
plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/boosting_test_q3b_precomputed_text",
    sample_name=sample_name,
    plot_config=plot_config,
)


In [ ]:
plot_averaged_layers(
    chunks_to_plot=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__previously_generating_tokens",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__im_start_0",
    ],
    base_path="<RUN_ROOT>/boosting_test_q3b_precomputed_image",
    sample_name=sample_name,
    plot_config=plot_config,
)


In [ ]:
#generate for all npz files in the directory
base_path = "<RUN_ROOT>/one_token_at_a_time/Rebuttal_MixedSignalsVSR/qwenv25vl7B/vsr_image_hard/"
for file_name in os.listdir(os.path.join(base_path+"/0/attention_progression/")):
    if file_name.endswith(".npz"):
        print(f"Processing {file_name}...")
        _, plt = plot_averaged_layers(
            chunks_to_plot=[
                "currently_generating_token__attends_to__image",
                "currently_generating_token__attends_to__text",
                "currently_generating_token__attends_to__previously_generating_tokens",
                "currently_generating_token__attends_to__instruction",
                "currently_generating_token__attends_to__im_start_0",
            ],
            base_path=base_path,
            sample_name=file_name,
            plot_config=plot_config,
        )
        #save in dir vsr_plot/vanilla with same name as npz but with .png extension
        save_dir = os.path.join("./vsr_plot", "image_hard_7b")
        print(f"Saving plot for {file_name} to {save_dir}...")
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, file_name.replace(".npz", ".png")))
        plt.close()